# Classic models (CNN) — vodič kroz notebook

Isti zadatak — prepoznavanje kompozitora po djelu — ali ovdje se ne pravi ručni feature vektor. Umjesto toga,
svaki audio segment se pretvara u **log-mel spektrogram** (2D "slika": mel frekvencija × vrijeme), i mala CNN
mreža sama uči šta u toj slici razlikuje kompozitore.

Segmenti su ovdje **10s** (`config.SEGMENT_DURATION`), kraći nego kod klasičnih modela (25s) — više trening
primjera za istu količinu audija, bitno kad imamo samo 113 djela.

PyTorch je CPU-only na ovoj mašini (nema CUDA) — trening je zato sporiji nego na GPU-u; model je namjerno
mali da to ostane izvodljivo.

In [8]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "..")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Ucitavanje podataka

## Priprema metapodataka

Pregled izvorne tabele i raspodjele kompozitora nalazi se u [metadata.ipynb](metadata.ipynb).
`load_metadata()` iz `dataset.py` učitava CSV, formira `work_id` kao `composer + " | " + composition`
i zadržava kompozitore sa najmanje `MIN_WORKS_PER_COMPOSER` različitih djela (podrazumijevano 5, iz `config.py`).
Svi stavovi istog djela tako ostaju u istoj grupi tokom evaluacije.

Ova sveska se izvršava samostalno; nije potrebno prethodno pokretati `metadata.ipynb`.

In [9]:
import numpy as np
import pandas as pd

from src.features.dataset import load_metadata
from src.utils.paths import AUDIO_DIR

metadata_filtered = load_metadata()

In [10]:
print(
    metadata_filtered.groupby("composer")["work_id"]
    .nunique()
    .sort_values()
)

composer
Brahms        8
Schubert      9
Mozart       11
Bach         30
Beethoven    55
Name: work_id, dtype: int64


---

## Mel-spektrogram dataset

`make_spectrogram_dataset` (u `spectrograms.py`) radi isto što i `make_dataset`/`make_midi_dataset` — učita
audio, isiječe na segmente, ali umjesto ručnih atributa računa `extract_melspec` (128 mel binova ×
~862 vremenskih frame-ova po segmentu, log-dB skala). Rezultat je keširan na disk (invalidira se automatski
ako se promijeni `spectrograms.py`).

Pošto su `SEGMENT_DURATION == MIN_LAST_DURATION == 10`, zadržavaju se samo puni 10s segmenti — svi imaju
identičnu dužinu, pa je `X` običan 3D numpy niz (N, 128, T), bez potrebe za paddingom.

In [11]:
from src.features.spectrograms import make_spectrogram_dataset

X, y, groups = make_spectrogram_dataset(metadata_filtered, AUDIO_DIR)

print(X.shape)

(11096, 64, 431)


---

# Model i trening

## CNN

Mali CNN (`SimpleCNN` u `nn_utils.py`): 3 conv+BN+ReLU+maxpool bloka, pa global average pooling i dropout
prije finalnog linearnog sloja — namjerno malo parametara zbog svega 113 djela. `SpecAugment` (nasumično
maskiranje frekvencijskih/vremenskih traka) je uključen samo na trening skupu, kao augmentacija.
`sample_weights` (isti mehanizam kao kod klasičnih modela) ide kroz `WeightedRandomSampler`, umjesto direktno
u loss funkciju.

In [ ]:
from src.model.nn_model import run_conv_experiment

cv_results, run_dir = run_conv_experiment(
    "cnn_small", X, y, groups,
    n_splits=5, epochs=15, batch_size=32, lr=1e-3, save_model=True
)

print("run_dir:", run_dir)


cnn_small | device: cpu
Classes: ['Bach' 'Beethoven' 'Brahms' 'Mozart' 'Schubert']

========== FOLD 1 ==========
epoch 5/15: train_loss=1.1592 train_acc=0.5170 val_loss=1.4317 val_acc=0.2381


## CRNN

Mali CRNN (`SimpleCRNN` u `nn_utils.py`): 2 conv+BN+ReLU+maxpool bloka, pa GRU sloj i dropout
prije finalnog linearnog sloja 

In [ ]:
from src.model.nn_model import run_conv_experiment

cv_results, run_dir = run_conv_experiment(
    "cnn_small", X, y, groups,
    n_splits=5, epochs=15, batch_size=32, lr=1e-3, save_model=True
)

print("run_dir:", run_dir)

---

## Work-level evaluacija

Model predviđa po segmentu; `soft_vote_predictions` (ista funkcija iz `evaluation.py` koju smo definisali za
klasične modele, do sad neiskorištena) prosječi softmax vjerovatnoće svih segmenata jednog djela i uzme
najvjerovatniju klasu — konzistentno sa `majority_vote_prediction` pristupom kod ostalih modela, samo
iskorišćava i "koliko je model siguran", ne samo tvrdu predikciju.